# Fabric Copilot SKU Estimator

Estimates Capacity Unit (CU) consumption for Power BI / Fabric Copilot usage and recommends a Fabric SKU based on expected user population, per-user activity, and peak concurrency.

## Billing Model
Per the Microsoft Fabric blog (*Announcing Copilot in Fabric pricing*, effective March 1, 2024):
- **Input (prompt)**: 400 CU-seconds per 1,000 tokens
- **Output (completion)**: 1,200 CU-seconds per 1,000 tokens

[Source: Microsoft Fabric Blog](https://blog.fabric.microsoft.com/en-us/blog/announcing-fabric-copilot-pricing-2/)

## How to use
1. Run cells 1 and 2 to load the constants and the `Estimate` class.
2. Run cell 4 — it will prompt you for your number of users, requests, and peak concurrent users, then print the recommended SKU.

In [ ]:
from dataclasses import dataclass
from typing import List, Optional

# ---- Pricing constants (CU-seconds per 1,000 tokens) ----
# Source: Microsoft Fabric blog, Feb 2024 announcement.
INPUT_RATE  = 400    # CU-sec / 1K input tokens
OUTPUT_RATE = 1200   # CU-sec / 1K output tokens

# ---- Fabric SKU catalog ----
SKUS = [
    ("F2",    2),
    ("F4",    4),
    ("F8",    8),
    ("F16",   16),
    ("F32",   32),
    ("F64",   64),    # equivalent to P1
    ("F128",  128),
    ("F256",  256),
    ("F512",  512),
    ("F1024", 1024),
    ("F2048", 2048),
]

In [ ]:
@dataclass
class Estimate:
    # Token shape of an average request
    input_tokens: int = 2000
    output_tokens: int = 500

    # Optional: original word counts (for display only — tokens drive the math).
    # Per the Fabric blog: 1,000 tokens ≈ 750 words.
    input_words: Optional[int] = None
    output_words: Optional[int] = None

    # User population & activity
    num_users: int = 100
    requests_per_user_per_day: float = 10.0

    # Concurrency model — set peak_concurrent_users explicitly, OR let it be
    # derived from concurrency_factor (fraction of num_users active at peak).
    concurrency_factor: float = 0.10
    peak_concurrent_users: Optional[int] = None

    # Average wall-clock seconds per Copilot request. Used to translate
    # in-flight concurrency into instantaneous CU/sec demand.
    avg_request_seconds: float = 6.0

    # ---------- Volume ----------
    @property
    def requests_per_day(self) -> int:
        return int(round(self.num_users * self.requests_per_user_per_day))

    # ---------- Per-request cost ----------
    @property
    def cu_seconds_per_request(self) -> float:
        return (self.input_tokens  * INPUT_RATE  / 1000.0
              + self.output_tokens * OUTPUT_RATE / 1000.0)

    # ---------- Daily totals ----------
    @property
    def cu_seconds_per_day(self) -> float:
        return self.cu_seconds_per_request * self.requests_per_day

    @property
    def cu_hours_per_day(self) -> float:
        return self.cu_seconds_per_day / 3600.0

    # ---------- Concurrency / peak ----------
    @property
    def effective_peak_concurrency(self) -> int:
        if self.peak_concurrent_users is not None:
            return max(1, int(self.peak_concurrent_users))
        return max(1, int(round(self.num_users * self.concurrency_factor)))

    @property
    def peak_cu_per_second(self) -> float:
        """Instantaneous CU/sec demand at peak concurrency."""
        if self.avg_request_seconds <= 0:
            return float("inf")
        return (self.effective_peak_concurrency
                * self.cu_seconds_per_request
                / self.avg_request_seconds)

    # ---------- Recommendation ----------
    def recommend_sku(self, headroom: float = 0.20) -> List[tuple]:
        """
        (name, cu, daily_cu_hours, fits_daily, fits_peak, fits_overall)
        Strict fit — ignores Fabric smoothing/bursting.
        """
        needed_daily = self.cu_hours_per_day * (1 + headroom)
        needed_peak  = self.peak_cu_per_second * (1 + headroom)
        rows = []
        for name, cu in SKUS:
            daily_budget = cu * 24
            fits_daily = daily_budget >= needed_daily
            fits_peak  = cu >= needed_peak
            rows.append((name, cu, daily_budget,
                         fits_daily, fits_peak,
                         fits_daily and fits_peak))
        return rows

    def report(self, headroom: float = 0.20) -> str:
        # ANSI escape codes: bold black on bright-yellow background
        BOLD_HL = "\033[1;30;103m"
        RESET   = "\033[0m"

        lines = []
        lines.append("Fabric Copilot SKU Estimator")
        lines.append("=" * 72)
        lines.append("Workload shape")
        if self.input_words is not None:
            lines.append(f"  Input  / request         : {self.input_words:,} words  ≈ {self.input_tokens:,} tokens")
        else:
            lines.append(f"  Input tokens / request   : {self.input_tokens:,}")
        if self.output_words is not None:
            lines.append(f"  Output / request         : {self.output_words:,} words  ≈ {self.output_tokens:,} tokens")
        else:
            lines.append(f"  Output tokens / request  : {self.output_tokens:,}")
        lines.append(f"  Avg request wall-time    : {self.avg_request_seconds:.1f} s")
        lines.append("")
        lines.append("Population & activity")
        lines.append(f"  Users                    : {self.num_users:,}")
        lines.append(f"  Requests / user / day    : {self.requests_per_user_per_day:,.2f}")
        lines.append(f"  Total requests / day     : {self.requests_per_day:,}")
        lines.append("")
        lines.append("Concurrency")
        if self.peak_concurrent_users is not None:
            lines.append(f"  Peak concurrent users    : {self.peak_concurrent_users:,} (explicit)")
        else:
            lines.append(f"  Concurrency factor       : {self.concurrency_factor*100:.1f}% of users")
            lines.append(f"  Peak concurrent users    : {self.effective_peak_concurrency:,} (derived)")
        lines.append("")
        lines.append("Computed demand")
        lines.append(f"  CU-seconds / request     : {self.cu_seconds_per_request:,.2f}")
        lines.append(f"  CU-minutes / request     : {self.cu_seconds_per_request/60:,.2f}")
        lines.append(f"  CU-hours / day (avg)     : {self.cu_hours_per_day:,.2f}")
        lines.append(f"  Peak CU/sec demand       : {self.peak_cu_per_second:,.2f}")
        lines.append(f"  Headroom buffer          : {headroom*100:.0f}%")
        lines.append("")
        lines.append(f"{'SKU':<8}{'CUs':>6}{'CU-hrs/day':>14}"
                     f"{'  Daily?':>10}{'  Peak?':>10}{'  Max req/day':>16}")
        lines.append("-" * 72)
        first_fit = None
        for name, cu, budget, fits_daily, fits_peak, fits in self.recommend_sku(headroom):
            max_req = int(budget * 3600 / self.cu_seconds_per_request)
            d = "YES" if fits_daily else "no"
            p = "YES" if fits_peak  else "no"
            if fits and first_fit is None:
                first_fit = name
            lines.append(f"{name:<8}{cu:>6}{budget:>14,}"
                         f"{d:>10}{p:>10}{max_req:>16,}")
        lines.append("")
        rec_text = (f" Recommended SKU (with {headroom*100:.0f}% headroom): "
                    f"{first_fit or 'exceeds F2048 — split workload'} ")
        lines.append(f"{BOLD_HL}{rec_text}{RESET}")
        return "\n".join(lines)

## SKU Recommendation

Edit the values at the top of the cell below — **number of users**, **requests per user per day**, and **peak concurrent users** — then run the cell. The estimator will compute CU demand and recommend the smallest Fabric SKU that fits with the chosen headroom buffer.

In [ ]:
# =====================================================================
# EDIT THESE VALUES, then run this cell to see the recommended SKU
# =====================================================================
num_users             = 100      # total licensed users
requests_per_user_day = 10       # avg Copilot calls / user / day
peak_concurrent_users = 10       # users active at the same time during peak

# Request size in WORDS (per Microsoft Fabric blog: 1,000 tokens ≈ 750 words)
input_words           = 750      # avg words in the prompt
output_words          = 188      # avg words in the response (~250 tokens)

# Optional tuning
avg_request_seconds   = 6.0      # avg request wall-clock seconds
headroom              = 0.20     # 20% headroom buffer
# =====================================================================

# Convert words to tokens (1,000 tokens ≈ 750 words  →  tokens = words * 1000/750)
WORDS_PER_1K_TOKENS = 750
input_tokens  = int(round(input_words  * 1000 / WORDS_PER_1K_TOKENS))
output_tokens = int(round(output_words * 1000 / WORDS_PER_1K_TOKENS))

workload = Estimate(
    input_tokens=input_tokens,
    output_tokens=output_tokens,
    input_words=input_words,
    output_words=output_words,
    num_users=num_users,
    requests_per_user_per_day=requests_per_user_day,
    peak_concurrent_users=peak_concurrent_users,
    avg_request_seconds=avg_request_seconds,
)

print(workload.report(headroom=headroom))